In [2]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import time
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, LongType, DoubleType, StringType
from pyspark.sql import Window


# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

S3_ENDPOINT = "http://minio:9000"
S3_BUCKET   = "s3a://warehouse"

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", S3_BUCKET)
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          S3_ENDPOINT)
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [3]:
# Looking at the taxi trip data
print(os.getcwd())
trips = spark.read.parquet("data/yellow_tripdata_2025-01.parquet")
trips.show(5)
print(trips)

/home/jovyan/project
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|               

In [4]:
# Loading the zone lookup table
print(os.getcwd())
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)
print(zones)

/home/jovyan/project
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
DataFrame[LocationID: bigint, Borough: string, Zone: string, service_zone: string]


In [ ]:
def reset_stream(query, checkpoint_path, table_name):
    # 1. Stop stream
    if query and query.isActive:
        query.stop()
        print("Stream stopped.")

    # 2. Drop table
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    print(f"Table '{table_name}' dropped.")

    # 3. Delete checkpoint
    try:
        shutil.rmtree(checkpoint_path, ignore_errors=True)
        print(f"Checkpoint at '{checkpoint_path}' deleted.")
    except Exception as e:
        print(f"Checkpoint deletion failed: {e}")

    # 4. Drop temp view
    spark.catalog.dropGlobalTempView("tmp_batch")
    print("Global temp view dropped.")

# reset_stream(query, "/tmp/chk-iceberg", "lakehouse.bronze.stg_taxi")

Table 'lakehouse.bronze.stg_taxi' dropped.
Checkpoint at '/tmp/chk-iceberg' deleted.
Global temp view dropped.


# Bronze layer

In [20]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [24]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze.stg_taxi (
        kafka_time TIMESTAMP,
        key STRING,
        offset INT,
        partition INT,
        value STRING
    ) USING iceberg
""")

parsed_df = raw_stream.select(
    F.col("key").cast("string").alias("key"),
    F.col("value").cast("string").alias("value"),
    F.col("partition").cast("int"),
    F.col("offset").cast("long"),
    F.col("timestamp").alias("kafka_time")
).select("key", "value", "partition", "offset", "kafka_time")

def write_to_iceberg(batch_df, batch_id):
    batch_df = batch_df.withColumn("kafka_time", F.to_timestamp("kafka_time"))

    batch_df.createOrReplaceGlobalTempView("tmp_batch")
    spark.sql("""
        MERGE INTO lakehouse.bronze.stg_taxi t
        USING global_temp.tmp_batch s
            ON t.partition = s.partition AND t.offset = s.offset
        WHEN NOT MATCHED THEN INSERT *
    """)
    # batch-level metrics
    batch_count = batch_df.count()
    if batch_count > 0:
        min_ts = batch_df.agg(F.min("kafka_time")).collect()[0][0]
        max_ts = batch_df.agg(F.max("kafka_time")).collect()[0][0]
    else:
        min_ts = None
        max_ts = None
    
    # log to stdout (visible in driver/executor logs)
    print(f"batch_id={batch_id} count={batch_count} min_kafka_time={min_ts} max_kafka_time={max_ts}")

query = (
    parsed_df.writeStream
      .foreachBatch(write_to_iceberg)
      .option("checkpointLocation", "/tmp/chk-iceberg")
      .trigger(processingTime="5 seconds")
      .start()
)

batch_id=8 count=438465 min_kafka_time=2026-04-02 15:08:00.005000 max_kafka_time=2026-04-02 15:17:21.171000
batch_id=9 count=2926 min_kafka_time=2026-04-02 15:17:21.172000 max_kafka_time=2026-04-02 15:17:25.002000
batch_id=10 count=3905 min_kafka_time=2026-04-02 15:17:25.004000 max_kafka_time=2026-04-02 15:17:30.002000


In [ ]:
# This is here for when you run all cells.
# Gives time to sync some of the data 
time.sleep(10)

In [25]:
# Run this to stop the query
query.stop()

## Testing Bronze layer

In [26]:
# Number of rows
df = spark.sql("""
    SELECT count(*) FROM lakehouse.bronze.stg_taxi
""")
print("Row count")
df.show(1)

# Checking for duplicates. Should return 0 rows
df = spark.sql("""
    SELECT key, offset, partition, count(*) AS c FROM lakehouse.bronze.stg_taxi
    GROUP BY key, offset, partition
    HAVING c > 1
""")

print("\nFollowing table show duplicates")
df.show(30)

Row count
+--------+
|count(1)|
+--------+
|  478470|
+--------+


Following table show duplicates
+---+------+---------+---+
|key|offset|partition|  c|
+---+------+---------+---+
+---+------+---------+---+



## Custom scenario

In [33]:
df = spark.sql("""
    SELECT partition, count(*) AS c FROM lakehouse.bronze.stg_taxi
    GROUP BY partition
    ORDER BY c desc
""")

print("\n Showing number of rows per partition")
df.show(30)

df = spark.sql("""
    SELECT key, partition, count(*) FROM lakehouse.bronze.stg_taxi
    group by 1,2 order by 1
""")

print("\n Since all the pickup locations are contained in their partitions, the ordering of taxi trips per pickup location is guaranteed")
df.show(30)


 Showing number of rows per partition
+---------+------+
|partition|     c|
+---------+------+
|        2|136710|
|        3|101380|
|        5| 78425|
|        4| 65497|
|        1| 53136|
|        0| 43322|
+---------+------+


 Since all the pickup locations are contained in their partitions, the ordering of taxi trips per pickup location is guaranteed
+---+---------+--------+
|key|partition|count(1)|
+---+---------+--------+
|  1|        3|     141|
| 10|        4|     214|
|100|        3|    7527|
|101|        2|      11|
|102|        3|      11|
|106|        0|      16|
|107|        3|    8808|
|108|        3|      28|
|109|        3|       1|
| 11|        0|      16|
|112|        5|      70|
|113|        4|    6104|
|114|        5|    7066|
|116|        2|     263|
|117|        5|      51|
|119|        3|      43|
| 12|        4|     338|
|120|        1|       2|
|121|        4|      20|
|122|        4|      12|
|123|        5|      23|
|124|        2|      21|
|125|        0| 

# Silver Layer

In [29]:
# SILVER LAYER
taxi_schema = StructType([
    StructField("VendorID", IntegerType()),
    StructField("tpep_pickup_datetime", TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count", IntegerType()),
    StructField("trip_distance", DoubleType()),
    StructField("RatecodeID", IntegerType()),
    StructField("store_and_fwd_flag", StringType()),
    StructField("PULocationID", IntegerType()),
    StructField("DOLocationID", IntegerType()),
    StructField("payment_type", IntegerType()),
    StructField("fare_amount", DoubleType()),
    StructField("extra", DoubleType()),
    StructField("mta_tax", DoubleType()),
    StructField("tip_amount", DoubleType()),
    StructField("tolls_amount", DoubleType()),
    StructField("improvement_surcharge", DoubleType()),
    StructField("total_amount", DoubleType()),
    StructField("congestion_surcharge", DoubleType()),
    StructField("Airport_fee", DoubleType()),
    StructField("cbd_congestion_fee", DoubleType())
])

spark.sql(f"""
    CREATE OR REPLACE TABLE lakehouse.silver.fct_taxi_trip (
        tripID STRING NOT NULL, 
        VendorID INT,
        RatecodeID INT,
        PULocationID INT,
        DOLocationID INT,
        tpep_pickup_datetime TIMESTAMP,
        tpep_dropoff_datetime TIMESTAMP,
        passenger_count INT,
        trip_distance DOUBLE,
        store_and_fwd_flag STRING,
        payment_type INT,
        fare_amount DOUBLE,
        extra DOUBLE,
        mta_tax DOUBLE,
        tip_amount DOUBLE,
        tolls_amount DOUBLE,
        improvement_surcharge DOUBLE,
        congestion_surcharge DOUBLE,
        Airport_fee DOUBLE,
        cbd_congestion_fee DOUBLE,
        total_amount DOUBLE,
        PU_Zone STRING,
        PU_Borough STRING,
        PU_service_zone STRING,
        DO_Zone STRING,
        DO_Borough STRING,
        DO_service_zone STRING
    ) USING iceberg
    TBLPROPERTIES (
        'write.identifier-columns' = 'tripID'
    )
""")

silver_df = (
    spark.table("lakehouse.bronze.stg_taxi")
    .select(F.from_json("value", taxi_schema).alias("d"))
    .select(
        F.col("d.VendorID").cast("int").alias("VendorID"),
        F.col("d.RatecodeID").cast("int").alias("RatecodeID"),
        F.col("d.PULocationID").cast("int").alias("PULocationID"),
        F.col("d.DOLocationID").cast("int").alias("DOLocationID"),
        F.to_timestamp(F.col("d.tpep_pickup_datetime")).alias("tpep_pickup_datetime"),
        F.to_timestamp(F.col("d.tpep_dropoff_datetime")).alias("tpep_dropoff_datetime"),
        F.col("d.passenger_count").cast("int").alias("passenger_count"),
        F.col("d.trip_distance").cast("double").alias("trip_distance"),
        F.col("d.store_and_fwd_flag").cast("string").alias("store_and_fwd_flag"),
        F.col("d.payment_type").cast("int").alias("payment_type"),
        F.col("d.fare_amount").cast("double").alias("fare_amount"),
        F.col("d.extra").cast("double").alias("extra"),
        F.col("d.mta_tax").cast("double").alias("mta_tax"),
        F.col("d.tip_amount").cast("double").alias("tip_amount"),
        F.col("d.tolls_amount").cast("double").alias("tolls_amount"),
        F.col("d.improvement_surcharge").cast("double").alias("improvement_surcharge"),
        F.col("d.congestion_surcharge").cast("double").alias("congestion_surcharge"),
        F.col("d.Airport_fee").cast("double").alias("Airport_fee"),
        F.col("d.cbd_congestion_fee").cast("double").alias("cbd_congestion_fee"),
        F.col("d.total_amount").cast("double").alias("total_amount"),
    )
    .na.fill({
        "passenger_count": 1,
        "trip_distance": 0.0,
        "store_and_fwd_flag": "N",
        "payment_type": 0,
        "fare_amount": 0.0,
        "extra": 0.0,
        "mta_tax": 0.0,
        "tip_amount": 0.0,
        "tolls_amount": 0.0,
        "improvement_surcharge": 0.0,
        "congestion_surcharge": 0.0,
        "Airport_fee": 0.0,
        "cbd_congestion_fee": 0.0,
        }
    )
)
# Creating a surrogate primary key
silver_df = silver_df.withColumn(
    "tripID",
    F.sha2(
        F.concat_ws("|",
            F.col("VendorID").cast("string"),
            F.col("tpep_pickup_datetime").cast("string"),
            F.col("tpep_dropoff_datetime").cast("string"),
            F.col("PULocationID").cast("string"),
            F.col("DOLocationID").cast("string"),
            F.col("total_amount").cast("string"),
        ),
        256  # SHA-256
    )
)

# For calculating the total amount in case that is missing
fee_components = (
    F.col("fare_amount") + F.col("extra") + F.col("mta_tax") + F.col("tip_amount") +
    F.col("tolls_amount") + F.col("improvement_surcharge") + F.col("congestion_surcharge") +
    F.col("Airport_fee") + F.col("cbd_congestion_fee")
)

silver_df = silver_df.select(
    *[c for c in silver_df.columns if c != "total_amount"],
    F.coalesce(F.col("total_amount"), fee_components).alias("total_amount")
)

# Deduplication
window = Window.partitionBy("tripID").orderBy(F.col("tpep_pickup_datetime").desc())

silver_df = (
    silver_df
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# joining in the zones
zones = spark.read.parquet("data/taxi_zone_lookup.parquet").select(
    F.col("LocationID").cast("int").alias("LocationID"),
    F.col("Zone"),
    F.col("Borough"),
    F.col("service_zone")
)

z_pu = zones.alias("z_pu")
z_do = zones.alias("z_do")
s = silver_df.alias("s")

silver_enriched = (
    s
    .join(F.broadcast(z_pu), F.col("s.PULocationID") == F.col("z_pu.LocationID"), "left")
    .join(F.broadcast(z_do), F.col("s.DOLocationID") == F.col("z_do.LocationID"), "left")
    .select(
        F.col("s.*"),
        F.col("z_pu.Zone").alias("PU_Zone"),
        F.col("z_pu.Borough").alias("PU_Borough"),
        F.col("z_pu.service_zone").alias("PU_service_zone"),
        F.col("z_do.Zone").alias("DO_Zone"),
        F.col("z_do.Borough").alias("DO_Borough"),
        F.col("z_do.service_zone").alias("DO_service_zone"),
    )
)

silver_enriched.writeTo("lakehouse.silver.fct_taxi_trip").append()

In [28]:
# Number of rows
df = spark.sql("""
    SELECT count(*) FROM lakehouse.silver.fct_taxi_trip
""")
print("Row count")
df.show(1)

# Looking at the data in bronze model
df = spark.sql("""
    SELECT * FROM lakehouse.silver.fct_taxi_trip WHERE tpep_dropoff_datetime = '2025-01-01 01:52:25'
""")
print("Following table shows examples")
df.show(10)


Row count
+--------+
|count(1)|
+--------+
|  478470|
+--------+

Following table shows examples
+--------------------+--------+----------+------------+------------+--------------------+---------------------+---------------+-------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------------+------------+------------+----------+---------------+--------------------+----------+---------------+
|              tripID|VendorID|RatecodeID|PULocationID|DOLocationID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|congestion_surcharge|Airport_fee|cbd_congestion_fee|total_amount|     PU_Zone|PU_Borough|PU_service_zone|             DO_Zone|DO_Borough|DO_service_zone|
+--------------------+--------+----------+------------+------------+--------------------+----------------

# Gold Layer

The goal is to find the most expensive trips by pickup location. By expensive we mean the most expensive per minute of ride

In [30]:
# Partitioning by months for faster querying, filtering when only looking at a specific time period
# Assuming in a real world case we would have more data than just January and February
# Partitioning by PU_Zone because this field is central to the table and will be used often
spark.sql("""
    CREATE OR REPLACE TABLE lakehouse.gold.analytical_taxi_trips
    USING ICEBERG
    PARTITIONED BY (months(tpep_pickup_datetime), bucket(16, PU_Zone))
    AS
    SELECT
      tpep_pickup_datetime,
      tpep_dropoff_datetime,
      PU_Zone,
      PU_Borough,
      PU_service_zone,
      total_amount,
      CAST(ROUND((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0) AS INT) AS trip_duration_minutes
    FROM lakehouse.silver.fct_taxi_trip
    WHERE tpep_pickup_datetime IS NOT NULL
      AND tpep_dropoff_datetime IS NOT NULL
""")

DataFrame[]

In [31]:
# Just inspecting the data
df = spark.sql("""
    SELECT * FROM lakehouse.gold.analytical_taxi_trips
""")
df.show(10)

# Finding the 10 most expensive PU zones in 2025 January

df = spark.sql("""
    SELECT 
        PU_Zone,
        ROUND(avg(total_amount / NULLIF(trip_duration_minutes, 0)), 2) AS avg_minute_fee,
        count(*) AS number_of_trips,
        ROUND(avg(trip_duration_minutes), 2) AS avg_trip_duration_minutes
    FROM lakehouse.gold.analytical_taxi_trips
    WHERE date_trunc('month', tpep_pickup_datetime) = '2025-01-01' 
    GROUP BY PU_Zone
    ORDER BY avg_minute_fee DESC
""")
df.show(10)

+--------------------+---------------------+-------------------+----------+---------------+------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|            PU_Zone|PU_Borough|PU_service_zone|total_amount|trip_duration_minutes|
+--------------------+---------------------+-------------------+----------+---------------+------------+---------------------+
| 2025-01-02 09:01:51|  2025-01-02 09:06:36|       Clinton West| Manhattan|    Yellow Zone|       13.44|                    5|
| 2025-01-04 20:28:46|  2025-01-04 21:05:54|Lincoln Square East| Manhattan|    Yellow Zone|        51.8|                   37|
| 2025-01-05 11:59:56|  2025-01-05 12:16:54|     Yorkville East| Manhattan|    Yellow Zone|        24.3|                   17|
| 2025-01-01 14:43:25|  2025-01-01 14:47:55|    Lenox Hill West| Manhattan|    Yellow Zone|        12.4|                    5|
| 2025-01-04 17:44:57|  2025-01-04 17:55:17|Lincoln Square East| Manhattan|    Yellow Zone|        15.4|       

In [32]:
# Iceberg snapshots
df = spark.sql("""
    SELECT * FROM lakehouse.gold.analytical_taxi_trips.snapshots;
""")
for row in df.collect():
    for col, val in row.asDict().items():
        print(f"{col}: {val}")
    print("-" * 40)

committed_at: 2026-03-28 18:36:54.699000
snapshot_id: 1665761210038241128
parent_id: None
operation: overwrite
manifest_list: s3://warehouse/gold/analytical_taxi_trips/metadata/snap-1665761210038241128-1-1de444b9-0ea4-4b22-8681-5b91e4740fe7.avro
summary: {'engine-version': '4.1.0', 'added-data-files': '4', 'total-equality-deletes': '0', 'app-id': 'local-1774691341551', 'added-records': '17683', 'total-records': '17683', 'spark.app.id': 'local-1774691341551', 'changed-partition-count': '1', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '223154', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.10.0 (commit 2114bf631e49af532d66e2ce148ee49dd1dd1f1f)', 'total-files-size': '223154', 'total-data-files': '4'}
----------------------------------------
committed_at: 2026-03-28 18:41:05.713000
snapshot_id: 6219255454133960955
parent_id: None
operation: overwrite
manifest_list: s3://warehouse/gold/analytical_taxi_trips/metadata/snap-6219255454133960955-1